# Matching INE--BANAVIM con criterios conservadores

Este notebook parte de las bases canónicas ya generadas en el notebook principal:

- `ine_fusion_2020_2022.csv`
- `banavim_fusion_2020_2022.csv.gz`

El objetivo no es repetir toda la limpieza, sino auditar los campos disponibles para record linkage y construir una estrategia de matching conservadora.

Dado que BANAVIM no contiene nombre del agresor ni un identificador común con INE, el matching no se interpreta como identificación individual de personas. Se interpreta como una vinculación aproximada entre registros o perfiles institucionales compatibles.

Antes de generar pares candidatos, se revisa especialmente la variable `vinculo_grupo`, porque el campo original `vinculo_victima` no tiene el mismo significado semántico en ambas fuentes.

In [1]:
import pandas as pd
import numpy as np
from pathlib import Path

OUTPUT_DIR = Path("../Data/fusion_outputs")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ine_fusion = pd.read_csv(
    OUTPUT_DIR / "ine_fusion_2020_2022.csv",
    encoding="utf-8-sig"
)

bv_fusion = pd.read_csv(
    OUTPUT_DIR / "banavim_fusion_2020_2022.csv.gz",
    encoding="utf-8-sig",
    compression="gzip"
)

print("ine_fusion:", ine_fusion.shape)
print("bv_fusion:", bv_fusion.shape)

display(ine_fusion.head())
display(bv_fusion.head())

ine_fusion: (162, 10)
bv_fusion: (806092, 10)


,id_ine_caso,Nombre,Número De Expediente,fecha_resolucion,anio_resolucion,sexo,estado,municipio,vinculo_victima,vinculo_grupo
0,INE_00000,AARÓN VEGA REA,SRE-PSC-41/2022,2022-04-07,2022,H,GUANAJUATO,NaN,NINGUNA,NINGUNA
1,INE_00001,ABEL TOVILLA CARPIO,IEPC/PE/GJMA/081/2021,2021-11-10,2021,H,CHIAPAS,TEOPISCA,PARES,PARES
2,INE_00002,ADÁN FRAUSTO ARELLANO,TEE-PES-18/2021,2021-07-02,2021,H,NAYARIT,EL NAYAR,JERARQUICA,JERARQUICA_SUBORDINACION
3,INE_00003,ADRIANA AVENDAÑO NIÑO,PES-58/2021,2021-06-04,2021,M,OAXACA,SAN ANDRES ZAUTLA,NINGUNA,NINGUNA
4,INE_00004,ALBERTO ALFONSO MENDOZA CRUZ,JDCI/52/2021 y su acumulado JDCI/56/2021,2021-07-09,2021,H,OAXACA,SAN LORENZO CACAOTEPEC,JERARQUICA,JERARQUICA_SUBORDINACION


,id_bv_caso,Identificador Único,año_base,fecha_registro,anio_registro,sexo,estado,municipio,vinculo_victima,vinculo_grupo
0,BV_0000000,0128900022-2,2020,2020-09-03 08:59:29,2020,H,AGUASCALIENTES,AGUASCALIENTES,EX PAREJA,PAREJA_EXPAREJA
1,BV_0000001,0128900106-2,2020,2020-10-20 09:25:05,2020,H,AGUASCALIENTES,AGUASCALIENTES,CONYUGE O PAREJA,PAREJA_EXPAREJA
2,BV_0000002,0128900132-2,2020,2020-07-02 08:07:24,2020,H,AGUASCALIENTES,AGUASCALIENTES,CONYUGE O PAREJA,PAREJA_EXPAREJA
3,BV_0000003,0128900350-2,2020,2020-01-29 15:48:51,2020,H,AGUASCALIENTES,AGUASCALIENTES,EX PAREJA,PAREJA_EXPAREJA
4,BV_0000004,0128900563-2,2020,2020-12-07 13:50:38,2020,H,AGUASCALIENTES,COSIO,CONYUGE O PAREJA,PAREJA_EXPAREJA


In [2]:
def auditar_origen_grupos(df, nombre_base):
    auditoria = (
        df.groupby(["vinculo_grupo", "vinculo_victima"], dropna=False)
        .size()
        .reset_index(name="frecuencia")
        .sort_values(["vinculo_grupo", "frecuencia"], ascending=[True, False])
    )
    auditoria.insert(0, "base", nombre_base)
    return auditoria

auditoria_vinculos_ine = auditar_origen_grupos(ine_fusion, "INE")
auditoria_vinculos_bv = auditar_origen_grupos(bv_fusion, "BANAVIM")

display(auditoria_vinculos_ine)
display(auditoria_vinculos_bv)

,base,vinculo_grupo,vinculo_victima,frecuencia
0,INE,INSTITUCIONAL_POLITICA,OPOSITOR EN LA CONTIENDA,2
1,INE,JERARQUICA_SUBORDINACION,JERARQUICA,33
2,INE,JERARQUICA_SUBORDINACION,SUBORDINACION,5
3,INE,NINGUNA,NINGUNA,57
5,INE,OTRO,OTRO,3
4,INE,OTRO,OTRA,1
6,INE,PARES,PARES,61


,base,vinculo_grupo,vinculo_victima,frecuencia
2,BANAVIM,COMUNITARIO_SOCIAL,VECINO A,11678
1,BANAVIM,COMUNITARIO_SOCIAL,PROFESOR A,549
0,BANAVIM,COMUNITARIO_SOCIAL,DESCONOCIDO,8
5,BANAVIM,FAMILIAR,HIJO A,23839
6,BANAVIM,FAMILIAR,MADRE O PADRE,23766
4,BANAVIM,FAMILIAR,HERMANO A,15065
8,BANAVIM,FAMILIAR,PADRASTRO O MADRASTRA,5272
12,BANAVIM,FAMILIAR,TIO A,5148
9,BANAVIM,FAMILIAR,PRIMO A,2671
10,BANAVIM,FAMILIAR,SOBRINO A,2468


## Revisión de categorías comparables desde INE hacia BANAVIM

Como el objetivo del matching es enriquecer o vincular registros del INE con BANAVIM, la revisión se centra primero en las categorías que efectivamente aparecen en la base del INE.

La pregunta de esta sección es:

> De las categorías de `vinculo_grupo` que aparecen en INE, ¿qué valores originales de BANAVIM fueron mapeados a esas mismas categorías?

Esto permite evaluar si las categorías compartidas son semánticamente suficientemente compatibles para usarse en matching conservador.

In [3]:
# Categorías de vinculo_grupo presentes en INE y BANAVIM

grupos_ine = set(ine_fusion["vinculo_grupo"].dropna().unique())
grupos_bv = set(bv_fusion["vinculo_grupo"].dropna().unique())

grupos_comunes = sorted(grupos_ine & grupos_bv)
grupos_solo_ine = sorted(grupos_ine - grupos_bv)
grupos_solo_bv = sorted(grupos_bv - grupos_ine)

resumen_grupos = pd.DataFrame({
    "tipo": [
        "grupos en INE",
        "grupos en BANAVIM",
        "grupos comunes",
        "grupos solo INE",
        "grupos solo BANAVIM"
    ],
    "n": [
        len(grupos_ine),
        len(grupos_bv),
        len(grupos_comunes),
        len(grupos_solo_ine),
        len(grupos_solo_bv)
    ],
    "grupos": [
        sorted(grupos_ine),
        sorted(grupos_bv),
        grupos_comunes,
        grupos_solo_ine,
        grupos_solo_bv
    ]
})

display(resumen_grupos)

,tipo,n,grupos
0,grupos en INE,5,"[INSTITUCIONAL_POLITICA, JERARQUICA_SUBORDINAC..."
1,grupos en BANAVIM,8,"[COMUNITARIO_SOCIAL, FAMILIAR, INSTITUCIONAL_P..."
2,grupos comunes,4,"[INSTITUCIONAL_POLITICA, JERARQUICA_SUBORDINAC..."
3,grupos solo INE,1,[NINGUNA]
4,grupos solo BANAVIM,4,"[COMUNITARIO_SOCIAL, FAMILIAR, MISSING, PAREJA..."


In [4]:
# Resumen completo:
# para cada vinculo_grupo creado en INE o BANAVIM,
# mostrar qué valores originales cayeron ahí en cada fuente.

def resumen_valores_por_grupo_todos(df, nombre_base):
    """
    Resume, por cada vinculo_grupo, los valores originales de vinculo_victima
    que cayeron en ese grupo.

    Importante:
    - No elimina missing originales.
    - Convierte NaN en una etiqueta explícita para no confundir:
      * grupo ausente en una fuente
      * valor original faltante dentro de un grupo existente
    """
    tmp = df.copy()

    tmp["vinculo_victima_mostrar"] = (
        tmp["vinculo_victima"]
        .astype("object")
        .where(tmp["vinculo_victima"].notna(), "<<MISSING_ORIGINAL>>")
    )

    resumen = (
        tmp.groupby("vinculo_grupo", dropna=False)
        .agg(
            registros=("vinculo_victima_mostrar", "size"),
            n_valores_originales=("vinculo_victima_mostrar", "nunique"),
            valores_originales=(
                "vinculo_victima_mostrar",
                lambda s: sorted(s.unique())
            )
        )
        .reset_index()
    )

    resumen = resumen.rename(columns={
        "registros": f"registros_{nombre_base}",
        "n_valores_originales": f"n_valores_originales_{nombre_base}",
        "valores_originales": f"valores_originales_{nombre_base}"
    })

    return resumen


resumen_grupos_ine = resumen_valores_por_grupo_todos(
    ine_fusion,
    "ine"
)

resumen_grupos_bv = resumen_valores_por_grupo_todos(
    bv_fusion,
    "banavim"
)

comparacion_todos_los_grupos = resumen_grupos_ine.merge(
    resumen_grupos_bv,
    on="vinculo_grupo",
    how="outer"
)

# Indicar en qué base aparece cada grupo
comparacion_todos_los_grupos["aparece_en_ine"] = (
    comparacion_todos_los_grupos["registros_ine"].notna()
)

comparacion_todos_los_grupos["aparece_en_banavim"] = (
    comparacion_todos_los_grupos["registros_banavim"].notna()
)

# Rellenar conteos numéricos, pero NO rellenar listas,
# porque NaN en listas significa "ese grupo no existe en esa fuente"
cols_num = [
    "registros_ine",
    "n_valores_originales_ine",
    "registros_banavim",
    "n_valores_originales_banavim"
]

comparacion_todos_los_grupos[cols_num] = (
    comparacion_todos_los_grupos[cols_num]
    .fillna(0)
    .astype(int)
)

# Reordenar columnas
comparacion_todos_los_grupos = comparacion_todos_los_grupos[
    [
        "vinculo_grupo",
        "aparece_en_ine",
        "aparece_en_banavim",
        "registros_ine",
        "n_valores_originales_ine",
        "valores_originales_ine",
        "registros_banavim",
        "n_valores_originales_banavim",
        "valores_originales_banavim"
    ]
].sort_values("vinculo_grupo")

display(comparacion_todos_los_grupos)

,vinculo_grupo,aparece_en_ine,aparece_en_banavim,registros_ine,n_valores_originales_ine,valores_originales_ine,registros_banavim,n_valores_originales_banavim,valores_originales_banavim
0,COMUNITARIO_SOCIAL,False,True,0,0,NaN,12235,3,"[DESCONOCIDO, PROFESOR A, VECINO A]"
1,FAMILIAR,False,True,0,0,NaN,82646,10,"[ABUELO A, HERMANO A, HIJO A, MADRE O PADRE, N..."
2,INSTITUCIONAL_POLITICA,True,True,2,1,[OPOSITOR EN LA CONTIENDA],515,1,[SERVIDOR PUBLICO]
3,JERARQUICA_SUBORDINACION,True,True,38,2,"[JERARQUICA, SUBORDINACION]",1927,1,[JEFE A O PATRON A]
4,MISSING,False,True,0,0,NaN,209481,2,"[<<MISSING_ORIGINAL>>, SELECCIONE]"
5,NINGUNA,True,False,57,1,[NINGUNA],0,0,NaN
6,OTRO,True,True,4,2,"[OTRA, OTRO]",31139,2,"[OTRA RELACION, OTRO]"
7,PAREJA_EXPAREJA,False,True,0,0,NaN,466234,4,"[CONCUBINA, CONYUGE O PAREJA, EX PAREJA, NOVIO A]"
8,PARES,True,True,61,1,[PARES],1915,1,[COMPANERO A]


In [5]:
# Valores originales que quedaron en REVISAR

def revisar_grupo(df, nombre_base, grupo="REVISAR"):
    tmp = df[df["vinculo_grupo"].eq(grupo)].copy()

    tmp["vinculo_victima_mostrar"] = (
        tmp["vinculo_victima"]
        .astype("object")
        .where(tmp["vinculo_victima"].notna(), "<<MISSING_ORIGINAL>>")
    )

    out = (
        tmp["vinculo_victima_mostrar"]
        .value_counts(dropna=False)
        .reset_index()
    )

    out.columns = ["valor_original", "frecuencia"]
    out.insert(0, "base", nombre_base)
    out.insert(1, "vinculo_grupo", grupo)

    return out


revisar_ine = revisar_grupo(ine_fusion, "INE", "REVISAR")
revisar_bv = revisar_grupo(bv_fusion, "BANAVIM", "REVISAR")

revisar_total = pd.concat(
    [revisar_ine, revisar_bv],
    ignore_index=True
)

display(revisar_total)

,base,vinculo_grupo,valor_original,frecuencia


In [6]:
# Valores originales que quedaron en MISSING

missing_ine = revisar_grupo(ine_fusion, "INE", "MISSING")
missing_bv = revisar_grupo(bv_fusion, "BANAVIM", "MISSING")

missing_total = pd.concat(
    [missing_ine, missing_bv],
    ignore_index=True
)

display(missing_total)

,base,vinculo_grupo,valor_original,frecuencia
0,BANAVIM,MISSING,<<MISSING_ORIGINAL>>,130927
1,BANAVIM,MISSING,SELECCIONE,78554


## Clasificación definitiva de vínculo para matching

Después de auditar los valores originales de `vinculo_victima`, se decidió que la clasificación operativa para matching no dependerá de la variable intermedia `vinculo_grupo`.

La variable `vinculo_grupo` fue útil durante la etapa de exploración y auditoría semántica, pero para el procedimiento final de matching se construye directamente una nueva variable:

- `vinculo_match`
- `confianza_vinculo_match`

Esta decisión simplifica la trazabilidad del notebook: el vínculo usado para matching se deriva directamente del valor original normalizado `vinculo_victima`, con reglas explícitas por fuente.

Se definieron tres niveles:

- `ALTA`: correspondencia semántica fuerte.
- `MEDIA`: correspondencia residual o débil, útil para revisión.
- `NO_APTO`: valores sin contraparte clara o no informativos.

Categorías de alta confianza:

| Fuente | Valor original | `vinculo_match` |
|---|---|---|
| INE | `PARES` | `PARES` |
| BANAVIM | `COMPANERO A` | `PARES` |
| INE | `JERARQUICA`, `SUBORDINACION` | `JERARQUICA_SUBORDINACION` |
| BANAVIM | `JEFE A O PATRON A` | `JERARQUICA_SUBORDINACION` |

Categorías de confianza media:

| Fuente | Valor original | `vinculo_match` |
|---|---|---|
| INE | `OTRO`, `OTRA` | `OTRO_RESIDUAL` |
| BANAVIM | `OTRO`, `OTRA RELACION` | `OTRO_RESIDUAL` |
| INE | `NINGUNA` | `SIN_RELACION_IDENTIFICABLE` |
| BANAVIM | `DESCONOCIDO` | `SIN_RELACION_IDENTIFICABLE` |

Las demás categorías se clasifican como `NO_APTO_MATCH`, porque no tienen una correspondencia suficientemente clara entre fuentes o no aportan información útil para el matching.

En las bases finales de matching no se conservará `vinculo_grupo`; se conservarán únicamente `vinculo_victima`, `vinculo_match` y `confianza_vinculo_match`.

In [7]:
# Variable operativa de vínculo para matching por niveles de confianza
# Versión definitiva: se construye directamente desde vinculo_victima,
# sin depender de vinculo_grupo.

def clasificar_vinculo_match(valor, fuente):
    """
    Clasifica el vínculo original normalizado hacia una categoría operativa
    para matching y un nivel de confianza.

    No usa vinculo_grupo. La clasificación se define directamente desde
    los valores originales normalizados de cada fuente.
    """
    if pd.isna(valor):
        return pd.Series({
            "vinculo_match": "NO_APTO_MATCH",
            "confianza_vinculo_match": "NO_APTO"
        })

    v = str(valor).strip().upper()

    # Placeholders o valores no informativos
    if v in {
        "",
        "NAN",
        "NONE",
        "SELECCIONE",
        "SIN DATO",
        "SIN DATOS",
        "NO ESPECIFICADO",
        "NO ESPECIFICADA",
        "NO SE ESPECIFICA"
    }:
        return pd.Series({
            "vinculo_match": "NO_APTO_MATCH",
            "confianza_vinculo_match": "NO_APTO"
        })

    # INE
    if fuente == "INE":
        if v == "PARES":
            return pd.Series({
                "vinculo_match": "PARES",
                "confianza_vinculo_match": "ALTA"
            })

        if v in {"JERARQUICA", "SUBORDINACION"}:
            return pd.Series({
                "vinculo_match": "JERARQUICA_SUBORDINACION",
                "confianza_vinculo_match": "ALTA"
            })

        if v in {"OTRO", "OTRA", "OTRA RELACION"}:
            return pd.Series({
                "vinculo_match": "OTRO_RESIDUAL",
                "confianza_vinculo_match": "MEDIA"
            })

        if v == "NINGUNA":
            return pd.Series({
                "vinculo_match": "SIN_RELACION_IDENTIFICABLE",
                "confianza_vinculo_match": "MEDIA"
            })

    # BANAVIM
    if fuente == "BANAVIM":
        if v in {"COMPANERO A", "COMPANERO", "COMPANERA"}:
            return pd.Series({
                "vinculo_match": "PARES",
                "confianza_vinculo_match": "ALTA"
            })

        if v in {"JEFE A O PATRON A", "JEFE A", "PATRON A", "JEFE", "PATRON"}:
            return pd.Series({
                "vinculo_match": "JERARQUICA_SUBORDINACION",
                "confianza_vinculo_match": "ALTA"
            })

        if v in {"OTRO", "OTRA", "OTRA RELACION", "OTRO TIPO DE RELACION"}:
            return pd.Series({
                "vinculo_match": "OTRO_RESIDUAL",
                "confianza_vinculo_match": "MEDIA"
            })

        if v == "DESCONOCIDO":
            return pd.Series({
                "vinculo_match": "SIN_RELACION_IDENTIFICABLE",
                "confianza_vinculo_match": "MEDIA"
            })

    # Todo lo demás no entra al matching operativo
    return pd.Series({
        "vinculo_match": "NO_APTO_MATCH",
        "confianza_vinculo_match": "NO_APTO"
    })


ine_fusion[["vinculo_match", "confianza_vinculo_match"]] = ine_fusion["vinculo_victima"].apply(
    lambda x: clasificar_vinculo_match(x, "INE")
)

bv_fusion[["vinculo_match", "confianza_vinculo_match"]] = bv_fusion["vinculo_victima"].apply(
    lambda x: clasificar_vinculo_match(x, "BANAVIM")
)

In [8]:
# Auditoría de la nueva clasificación operativa, sin usar vinculo_grupo

def auditoria_vinculo_match(df, nombre_base):
    out = (
        df.groupby(
            ["vinculo_match", "confianza_vinculo_match", "vinculo_victima"],
            dropna=False
        )
        .size()
        .reset_index(name="frecuencia")
        .sort_values(
            ["confianza_vinculo_match", "vinculo_match", "frecuencia"],
            ascending=[True, True, False]
        )
    )
    out.insert(0, "base", nombre_base)
    return out


auditoria_vinculo_match_total = pd.concat(
    [
        auditoria_vinculo_match(ine_fusion, "INE"),
        auditoria_vinculo_match(bv_fusion, "BANAVIM")
    ],
    ignore_index=True
)

display(auditoria_vinculo_match_total)

,base,vinculo_match,confianza_vinculo_match,vinculo_victima,frecuencia
0,INE,JERARQUICA_SUBORDINACION,ALTA,JERARQUICA,33
1,INE,JERARQUICA_SUBORDINACION,ALTA,SUBORDINACION,5
2,INE,PARES,ALTA,PARES,61
3,INE,OTRO_RESIDUAL,MEDIA,OTRO,3
4,INE,OTRO_RESIDUAL,MEDIA,OTRA,1
5,INE,SIN_RELACION_IDENTIFICABLE,MEDIA,NINGUNA,57
6,INE,NO_APTO_MATCH,NO_APTO,OPOSITOR EN LA CONTIENDA,2
7,BANAVIM,JERARQUICA_SUBORDINACION,ALTA,JEFE A O PATRON A,1927
8,BANAVIM,PARES,ALTA,COMPANERO A,1915
9,BANAVIM,OTRO_RESIDUAL,MEDIA,OTRO,31138


## Construcción de bases por nivel de confianza y revisión de municipios

A partir de la clasificación definitiva de `vinculo_victima`, se generan dos escenarios de bases para matching:

- **Alta confianza**: conserva únicamente categorías con equivalencia semántica fuerte.
- **Alta + media confianza**: incorpora además categorías residuales o débiles que podrían ser útiles para revisión individual.

Antes de ejecutar el procedimiento de record linkage, se exportan estas bases para inspección manual y se auditan los municipios disponibles en cada fuente.

La revisión se realiza en dos niveles:

1. `municipio`, para observar coincidencias textuales generales.
2. `estado + municipio`, para evitar confundir municipios homónimos en distintos estados.

Esta segunda revisión es la más importante para decidir si los campos geográficos pueden usarse como parte del matching.

In [9]:
# Crear bases para revisión/matching por nivel de confianza

campos_obligatorios_match = [
    "sexo",
    "estado",
    "municipio",
    "vinculo_match",
    "confianza_vinculo_match"
]

def filtrar_base_match(df, niveles_confianza):
    """
    Filtra registros aptos para matching según:
    - nivel de confianza del vínculo;
    - campos mínimos completos.
    """
    mask = (
        df["confianza_vinculo_match"].isin(niveles_confianza)
        & df["sexo"].notna()
        & df["estado"].notna()
        & df["municipio"].notna()
        & df["vinculo_match"].notna()
    )
    return df.loc[mask].copy()


# Escenario estricto: solo confianza alta
ine_match_alta = filtrar_base_match(
    ine_fusion,
    niveles_confianza=["ALTA"]
)

bv_match_alta = filtrar_base_match(
    bv_fusion,
    niveles_confianza=["ALTA"]
)


# Escenario ampliado: confianza alta + media
ine_match_alta_media = filtrar_base_match(
    ine_fusion,
    niveles_confianza=["ALTA", "MEDIA"]
)

bv_match_alta_media = filtrar_base_match(
    bv_fusion,
    niveles_confianza=["ALTA", "MEDIA"]
)


resumen_bases_match = pd.DataFrame([
    {
        "escenario": "ALTA",
        "base": "INE",
        "registros": len(ine_match_alta),
        "porcentaje_sobre_base_original": round(len(ine_match_alta) / len(ine_fusion) * 100, 2)
    },
    {
        "escenario": "ALTA",
        "base": "BANAVIM",
        "registros": len(bv_match_alta),
        "porcentaje_sobre_base_original": round(len(bv_match_alta) / len(bv_fusion) * 100, 2)
    },
    {
        "escenario": "ALTA_MEDIA",
        "base": "INE",
        "registros": len(ine_match_alta_media),
        "porcentaje_sobre_base_original": round(len(ine_match_alta_media) / len(ine_fusion) * 100, 2)
    },
    {
        "escenario": "ALTA_MEDIA",
        "base": "BANAVIM",
        "registros": len(bv_match_alta_media),
        "porcentaje_sobre_base_original": round(len(bv_match_alta_media) / len(bv_fusion) * 100, 2)
    }
])

display(resumen_bases_match)

,escenario,base,registros,porcentaje_sobre_base_original
0,ALTA,INE,98,60.49
1,ALTA,BANAVIM,3824,0.47
2,ALTA_MEDIA,INE,128,79.01
3,ALTA_MEDIA,BANAVIM,34782,4.31


In [10]:
# Preparar bases finales para exportación
# Se elimina vinculo_grupo porque ya no forma parte de la clasificación definitiva.

columnas_a_quitar_export = [
    "vinculo_grupo"
]

def preparar_export_match(df):
    cols_quitar = [c for c in columnas_a_quitar_export if c in df.columns]
    return df.drop(columns=cols_quitar).copy()


ine_match_alta_export = preparar_export_match(ine_match_alta)
bv_match_alta_export = preparar_export_match(bv_match_alta)

ine_match_alta_media_export = preparar_export_match(ine_match_alta_media)
bv_match_alta_media_export = preparar_export_match(bv_match_alta_media)


# Verificación
for nombre, df in [
    ("ine_match_alta_export", ine_match_alta_export),
    ("bv_match_alta_export", bv_match_alta_export),
    ("ine_match_alta_media_export", ine_match_alta_media_export),
    ("bv_match_alta_media_export", bv_match_alta_media_export)
]:
    print(nombre, df.shape)
    print("¿Tiene vinculo_grupo?", "vinculo_grupo" in df.columns)
    print()

ine_match_alta_export (98, 11)
¿Tiene vinculo_grupo? False

bv_match_alta_export (3824, 11)
¿Tiene vinculo_grupo? False

ine_match_alta_media_export (128, 11)
¿Tiene vinculo_grupo? False

bv_match_alta_media_export (34782, 11)
¿Tiene vinculo_grupo? False



## Validación geográfica contra catálogo INEGI

Después de construir las bases de matching por nivel de confianza, se realizó una validación adicional de los campos geográficos.

La revisión anterior comparaba municipios entre INE y BANAVIM. En esta sección el enfoque cambia: ahora se valida cada par `estado + municipio` contra el catálogo oficial de municipios de INEGI.

Esta validación es necesaria porque el nombre del municipio por sí solo no es suficiente para confirmar equivalencias. Existen municipios homónimos, nombres muy parecidos en una misma entidad y posibles variantes de escritura. Por ello, la validación se hace siempre a nivel `estado + municipio`.

Los municipios que no coinciden exactamente con el catálogo de INEGI no se tratan automáticamente como errores. En esta etapa se clasifican como registros que requieren revisión, ya que pueden corresponder a:

- errores de escritura;
- variantes no oficiales de un nombre municipal;
- localidades u otros lugares que no son municipios;
- valores antiguos o no homologados;
- registros que deben conservarse para análisis descriptivo, pero no usarse como evidencia geográfica fuerte en el matching.

El objetivo de esta sección es generar una auditoría reproducible antes del record linkage, para evitar que la vinculación entre fuentes dependa de municipios mal escritos o no validados.

In [11]:
# Guardar bases de matching para revisión previa

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

ine_match_alta_export.to_csv(
    OUTPUT_DIR / "ine_match_alta_confianza_2020_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

bv_match_alta_export.to_csv(
    OUTPUT_DIR / "banavim_match_alta_confianza_2020_2022.csv.gz",
    index=False,
    encoding="utf-8-sig",
    compression="gzip"
)

ine_match_alta_media_export.to_csv(
    OUTPUT_DIR / "ine_match_alta_media_confianza_2020_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

bv_match_alta_media_export.to_csv(
    OUTPUT_DIR / "banavim_match_alta_media_confianza_2020_2022.csv.gz",
    index=False,
    encoding="utf-8-sig",
    compression="gzip"
)

# Muestras ligeras para Data Wrangler
bv_match_alta_export.sample(
    n=min(50000, len(bv_match_alta_export)),
    random_state=42
).to_csv(
    OUTPUT_DIR / "banavim_match_alta_confianza_sample_50000.csv",
    index=False,
    encoding="utf-8-sig"
)

bv_match_alta_media_export.sample(
    n=min(50000, len(bv_match_alta_media_export)),
    random_state=42
).to_csv(
    OUTPUT_DIR / "banavim_match_alta_media_confianza_sample_50000.csv",
    index=False,
    encoding="utf-8-sig"
)

resumen_bases_match.to_csv(
    OUTPUT_DIR / "resumen_bases_match_confianza.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Bases guardadas en:", OUTPUT_DIR)

Bases guardadas en: ../Data/fusion_outputs


In [15]:
# Agregar columna de estado + municipio a todas las bases de matching

def agregar_estado_municipio(df):
    """
    Crea una columna combinada 'estado_municipio' a partir de 'estado' y 'municipio'.
    """
    df_copy = df.copy()
    df_copy["estado_municipio"] = df_copy["estado"] + " - " + df_copy["municipio"]
    return df_copy


# Actualizar todas las bases exportables
ine_match_alta_export = agregar_estado_municipio(ine_match_alta_export)
bv_match_alta_export = agregar_estado_municipio(bv_match_alta_export)

ine_match_alta_media_export = agregar_estado_municipio(ine_match_alta_media_export)
bv_match_alta_media_export = agregar_estado_municipio(bv_match_alta_media_export)


# Guardar bases actualizadas
ine_match_alta_export.to_csv(
    OUTPUT_DIR / "ine_match_alta_confianza_2020_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

bv_match_alta_export.to_csv(
    OUTPUT_DIR / "banavim_match_alta_confianza_2020_2022.csv.gz",
    index=False,
    encoding="utf-8-sig",
    compression="gzip"
)

ine_match_alta_media_export.to_csv(
    OUTPUT_DIR / "ine_match_alta_media_confianza_2020_2022.csv",
    index=False,
    encoding="utf-8-sig"
)

bv_match_alta_media_export.to_csv(
    OUTPUT_DIR / "banavim_match_alta_media_confianza_2020_2022.csv.gz",
    index=False,
    encoding="utf-8-sig",
    compression="gzip"
)

# Actualizar muestras ligeras
bv_match_alta_export.sample(
    n=min(50000, len(bv_match_alta_export)),
    random_state=42
).to_csv(
    OUTPUT_DIR / "banavim_match_alta_confianza_sample_50000.csv",
    index=False,
    encoding="utf-8-sig"
)

bv_match_alta_media_export.sample(
    n=min(50000, len(bv_match_alta_media_export)),
    random_state=42
).to_csv(
    OUTPUT_DIR / "banavim_match_alta_media_confianza_sample_50000.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Bases actualizadas con columna 'estado_municipio' guardadas en:", OUTPUT_DIR)

Bases actualizadas con columna 'estado_municipio' guardadas en: ../Data/fusion_outputs


In [17]:
# Auditoría de municipios disponibles por escenario de confianza

def auditar_estado_municipio_simple(df_ine, df_bv, escenario):
    municipios_ine = sorted(df_ine["estado_municipio"].dropna().unique())
    municipios_bv = sorted(df_bv["estado_municipio"].dropna().unique())

    comunes = sorted(set(municipios_ine) & set(municipios_bv))
    solo_ine = sorted(set(municipios_ine) - set(municipios_bv))
    solo_bv = sorted(set(municipios_bv) - set(municipios_ine))

    resumen = pd.DataFrame([
        {
            "escenario": escenario,
            "tipo": "estado_municipio INE",
            "n": len(municipios_ine)
        },
        {
            "escenario": escenario,
            "tipo": "estado_municipio BANAVIM",
            "n": len(municipios_bv)
        },
        {
            "escenario": escenario,
            "tipo": "estado_municipio comunes",
            "n": len(comunes)
        },
        {
            "escenario": escenario,
            "tipo": "estado_municipio solo INE",
            "n": len(solo_ine)
        },
        {
            "escenario": escenario,
            "tipo": "estado_municipio solo BANAVIM",
            "n": len(solo_bv)
        }
    ])

    detalle = pd.DataFrame(
        [{"escenario": escenario, "tipo": "comun", "estado_municipio": m} for m in comunes]
        + [{"escenario": escenario, "tipo": "solo_ine", "estado_municipio": m} for m in solo_ine]
        + [{"escenario": escenario, "tipo": "solo_banavim", "estado_municipio": m} for m in solo_bv]
    )

    return resumen, detalle


resumen_estado_municipio_alta, detalle_estado_municipio_alta = auditar_estado_municipio_simple(
    ine_match_alta_export,
    bv_match_alta_export,
    "ALTA"
)

resumen_estado_municipio_alta_media, detalle_estado_municipio_alta_media = auditar_estado_municipio_simple(
    ine_match_alta_media_export,
    bv_match_alta_media_export,
    "ALTA_MEDIA"
)

resumen_estado_municipio_match = pd.concat(
    [resumen_estado_municipio_alta, resumen_estado_municipio_alta_media],
    ignore_index=True
)

detalle_estado_municipio_match = pd.concat(
    [detalle_estado_municipio_alta, detalle_estado_municipio_alta_media],
    ignore_index=True
)

display(resumen_estado_municipio_match)
display(detalle_estado_municipio_match)

,escenario,tipo,n
0,ALTA,estado_municipio INE,40
1,ALTA,estado_municipio BANAVIM,471
2,ALTA,estado_municipio comunes,10
3,ALTA,estado_municipio solo INE,30
4,ALTA,estado_municipio solo BANAVIM,461
5,ALTA_MEDIA,estado_municipio INE,59
6,ALTA_MEDIA,estado_municipio BANAVIM,1116
7,ALTA_MEDIA,estado_municipio comunes,34
8,ALTA_MEDIA,estado_municipio solo INE,25
9,ALTA_MEDIA,estado_municipio solo BANAVIM,1082


,escenario,tipo,estado_municipio
0,ALTA,comun,CHIAPAS - CATAZAJA
1,ALTA,comun,CHIAPAS - TEOPISCA
2,ALTA,comun,CHIAPAS - TUZANTAN
3,ALTA,comun,COAHUILA - FRANCISCO I MADERO
4,ALTA,comun,HIDALGO - ZIMAPAN
...,...,...,...
1637,ALTA_MEDIA,solo_banavim,ZACATECAS - VILLA DE COS
1638,ALTA_MEDIA,solo_banavim,ZACATECAS - VILLA GARCIA
1639,ALTA_MEDIA,solo_banavim,ZACATECAS - VILLA GONZALEZ ORTEGA
1640,ALTA_MEDIA,solo_banavim,ZACATECAS - VILLANUEVA


In [18]:
resumen_estado_municipio_match.to_csv(
    OUTPUT_DIR / "resumen_estado_municipio_match_confianza.csv",
    index=False,
    encoding="utf-8-sig"
)

detalle_estado_municipio_match.to_csv(
    OUTPUT_DIR / "detalle_estado_municipio_match_confianza.csv",
    index=False,
    encoding="utf-8-sig"
)

In [21]:
# Construir tabla de pares únicos de estado y municipio por base y escenario

pares_estado_municipio = pd.concat([
    ine_match_alta_export[["estado", "municipio"]].drop_duplicates().assign(base="INE", escenario="ALTA"),
    bv_match_alta_export[["estado", "municipio"]].drop_duplicates().assign(base="BANAVIM", escenario="ALTA"),
    ine_match_alta_media_export[["estado", "municipio"]].drop_duplicates().assign(base="INE", escenario="ALTA_MEDIA"),
    bv_match_alta_media_export[["estado", "municipio"]].drop_duplicates().assign(base="BANAVIM", escenario="ALTA_MEDIA")
], ignore_index=True)

# Eliminar nulos en estado y municipio
pares_estado_municipio = pares_estado_municipio.dropna(subset=["estado", "municipio"])

# Eliminar duplicados completos
pares_estado_municipio = pares_estado_municipio.drop_duplicates()

# Ordenar por escenario, base, estado y municipio
pares_estado_municipio = pares_estado_municipio.sort_values(
    ["escenario", "base", "estado", "municipio"]
).reset_index(drop=True)

display(pares_estado_municipio)
print(f"\nTotal de pares únicos: {len(pares_estado_municipio)}")

,estado,municipio,base,escenario
0,AGUASCALIENTES,AGUASCALIENTES,BANAVIM,ALTA
1,AGUASCALIENTES,ASIENTOS,BANAVIM,ALTA
2,AGUASCALIENTES,CALVILLO,BANAVIM,ALTA
3,AGUASCALIENTES,COSIO,BANAVIM,ALTA
4,AGUASCALIENTES,EL LLANO,BANAVIM,ALTA
...,...,...,...,...
1681,VERACRUZ,CORDOBA,INE,ALTA_MEDIA
1682,VERACRUZ,LA ANTIGUA,INE,ALTA_MEDIA
1683,VERACRUZ,LERDO DE TEJADA,INE,ALTA_MEDIA
1684,VERACRUZ,PEROTE,INE,ALTA_MEDIA



Total de pares únicos: 1686


In [22]:
import unicodedata
import re

def normalizar_geo(s):
    """
    Normaliza una cadena geográfica:
    - Convierte a mayúsculas
    - Elimina acentos
    - Elimina puntuación
    - Colapsa espacios múltiples
    - Devuelve texto limpio
    """
    if pd.isna(s):
        return ""
    
    s = str(s).strip().upper()
    
    # Eliminar acentos
    s = "".join(
        c for c in unicodedata.normalize("NFD", s)
        if unicodedata.category(c) != "Mn"
    )
    
    # Eliminar puntuación (excepto espacios)
    s = re.sub(r"[^\w\s]", "", s)
    
    # Colapsar espacios múltiples
    s = re.sub(r"\s+", " ", s).strip()
    
    return s


# Aplicar normalización a pares_estado_municipio
pares_estado_municipio["estado_norm"] = pares_estado_municipio["estado"].apply(normalizar_geo)
pares_estado_municipio["municipio_norm"] = pares_estado_municipio["municipio"].apply(normalizar_geo)

# Muestra de resultados
display(pares_estado_municipio[["estado", "municipio", "estado_norm", "municipio_norm"]].head(15))

# Contar pares únicos
pares_unicos = pares_estado_municipio[["estado_norm", "municipio_norm"]].drop_duplicates()
print(f"\nTotal de pares únicos normalizados: {len(pares_unicos)}")
print(f"Total de filas antes de deduplicación: {len(pares_estado_municipio)}")

,estado,municipio,estado_norm,municipio_norm
0,AGUASCALIENTES,AGUASCALIENTES,AGUASCALIENTES,AGUASCALIENTES
1,AGUASCALIENTES,ASIENTOS,AGUASCALIENTES,ASIENTOS
2,AGUASCALIENTES,CALVILLO,AGUASCALIENTES,CALVILLO
3,AGUASCALIENTES,COSIO,AGUASCALIENTES,COSIO
4,AGUASCALIENTES,EL LLANO,AGUASCALIENTES,EL LLANO
5,AGUASCALIENTES,JESUS MARIA,AGUASCALIENTES,JESUS MARIA
6,AGUASCALIENTES,PABELLON DE ARTEAGA,AGUASCALIENTES,PABELLON DE ARTEAGA
7,AGUASCALIENTES,RINCON DE ROMOS,AGUASCALIENTES,RINCON DE ROMOS
8,AGUASCALIENTES,SAN FRANCISCO DE LOS ROMO,AGUASCALIENTES,SAN FRANCISCO DE LOS ROMO
9,AGUASCALIENTES,SAN JOSE DE GRACIA,AGUASCALIENTES,SAN JOSE DE GRACIA



Total de pares únicos normalizados: 1141
Total de filas antes de deduplicación: 1686


In [23]:
# Auditoría del efecto de normalización geográfica en pares_estado_municipio

# 1. Número total de filas
total_filas = len(pares_estado_municipio)

# 2. Pares únicos originales
pares_originales_unicos = pares_estado_municipio[["estado", "municipio"]].drop_duplicates()
n_pares_originales = len(pares_originales_unicos)

# 3. Pares únicos normalizados
pares_normalizados_unicos = pares_estado_municipio[["estado_norm", "municipio_norm"]].drop_duplicates()
n_pares_normalizados = len(pares_normalizados_unicos)

# 4. Filas que cambiaron textualmente
pares_estado_municipio["cambio_detectable"] = (
    (pares_estado_municipio["estado"] != pares_estado_municipio["estado_norm"]) |
    (pares_estado_municipio["municipio"] != pares_estado_municipio["municipio_norm"])
)
n_filas_con_cambios = pares_estado_municipio["cambio_detectable"].sum()

# Resumen general
resumen_normalizacion_geo = pd.DataFrame([
    {
        "metrica": "Total de filas",
        "valor": total_filas
    },
    {
        "metrica": "Pares únicos originales (estado, municipio)",
        "valor": n_pares_originales
    },
    {
        "metrica": "Pares únicos normalizados (estado_norm, municipio_norm)",
        "valor": n_pares_normalizados
    },
    {
        "metrica": "Filas con cambios detectables",
        "valor": n_filas_con_cambios
    },
    {
        "metrica": "Porcentaje de filas con cambios",
        "valor": f"{round(n_filas_con_cambios / total_filas * 100, 2)}%"
    }
])

display(resumen_normalizacion_geo)

# 5. Ejemplos de filas donde hubo cambios
cambios_normalizacion_geo = pares_estado_municipio[
    pares_estado_municipio["cambio_detectable"]
][["estado", "municipio", "estado_norm", "municipio_norm", "base", "escenario"]].drop_duplicates().head(20)

display(cambios_normalizacion_geo)

# 6. Detectar colisiones: varios pares originales -> mismo par normalizado
pares_con_orig = pares_estado_municipio[[
    "estado", "municipio", "estado_norm", "municipio_norm", "base", "escenario"
]].drop_duplicates()

# Agrupar por par normalizado y contar cuántos originales mapean a él
colisiones = pares_con_orig.groupby(["estado_norm", "municipio_norm"]).agg({
    "estado": lambda x: list(set(x)),
    "municipio": lambda x: list(set(x))
}).reset_index()

colisiones["n_originales_estado"] = colisiones["estado"].apply(len)
colisiones["n_originales_municipio"] = colisiones["municipio"].apply(len)
colisiones["tiene_colision"] = (colisiones["n_originales_estado"] > 1) | (colisiones["n_originales_municipio"] > 1)

colisiones_detectadas = colisiones[colisiones["tiene_colision"]].drop(columns=["estado", "municipio"])

if len(colisiones_detectadas) > 0:
    display(colisiones_detectadas)
else:
    print("No se detectaron colisiones de normalización.")

,metrica,valor
0,Total de filas,1686
1,"Pares únicos originales (estado, municipio)",1141
2,"Pares únicos normalizados (estado_norm, munici...",1141
3,Filas con cambios detectables,0
4,Porcentaje de filas con cambios,0.0%


,estado,municipio,estado_norm,municipio_norm,base,escenario


No se detectaron colisiones de normalización.


La celda anterior nos dice que la función de normalización construida para comparar de la misma forma a la base del INEGI y la que estamos trabajando, no hizo nada sobre nuestra base porque ya estaba preprocesada.

In [27]:
CVE_ENT = {
    "AGUASCALIENTES": "01",
    "BAJA CALIFORNIA": "02",
    "BAJA CALIFORNIA SUR": "03",
    "CAMPECHE": "04",
    "COAHUILA": "05",
    "COLIMA": "06",
    "CHIAPAS": "07",
    "CHIHUAHUA": "08",
    "CDMX": "09",
    "DURANGO": "10",
    "GUANAJUATO": "11",
    "GUERRERO": "12",
    "HIDALGO": "13",
    "JALISCO": "14",
    "MEXICO": "15",
    "MICHOACAN": "16",
    "MORELOS": "17",
    "NAYARIT": "18",
    "NUEVO LEON": "19",
    "OAXACA": "20",
    "PUEBLA": "21",
    "QUERETARO": "22",
    "QUINTANA ROO": "23",
    "SAN LUIS POTOSI": "24",
    "SINALOA": "25",
    "SONORA": "26",
    "TABASCO": "27",
    "TAMAULIPAS": "28",
    "TLAXCALA": "29",
    "VERACRUZ": "30",
    "YUCATAN": "31",
    "ZACATECAS": "32",
}

In [28]:
import requests
import time

catalogos_municipios = []

estados_a_consultar = sorted(pares_estado_municipio["estado_norm"].dropna().unique())

for estado_norm in estados_a_consultar:
    cve_ent = CVE_ENT.get(estado_norm)
    
    if cve_ent is None:
        print(f"Advertencia: no se encontró clave para {estado_norm}")
        continue
    
    url = f"https://gaia.inegi.org.mx/wscatgeo/v2/mgem/{cve_ent}"
    
    try:
        response = requests.get(url, timeout=30)
        response.raise_for_status()
        data = response.json()
        
        # Según tu diagnóstico, la respuesta viene como dict con llave "datos"
        if isinstance(data, dict) and "datos" in data:
            municipios = data["datos"]
        elif isinstance(data, list):
            municipios = data
        else:
            municipios = []
        
        for mun in municipios:
            catalogos_municipios.append({
                "estado": estado_norm,
                "cve_ent": mun.get("cve_ent"),
                "cve_mun": mun.get("cve_mun"),
                "cvegeo": mun.get("cvegeo"),
                "municipio_inegi": mun.get("nomgeo"),
                "cabecera_inegi": mun.get("nom_cab"),
                "pob_total": mun.get("pob_total")
            })
        
        print(f"✓ {estado_norm}: {len(municipios)} municipios descargados")
        
    except Exception as e:
        print(f"✗ Error consultando {estado_norm}: {e}")
    
    time.sleep(0.2)


catalogo_inegi_municipios = pd.DataFrame(catalogos_municipios)

catalogo_inegi_municipios["estado_norm"] = catalogo_inegi_municipios["estado"].apply(normalizar_geo)
catalogo_inegi_municipios["municipio_inegi_norm"] = catalogo_inegi_municipios["municipio_inegi"].apply(normalizar_geo)

display(catalogo_inegi_municipios.head())
print("Shape:", catalogo_inegi_municipios.shape)

display(
    catalogo_inegi_municipios
    .groupby("estado")
    .size()
    .reset_index(name="n_municipios")
)

print("Municipios INEGI faltantes:")
print(catalogo_inegi_municipios["municipio_inegi"].isna().sum())

catalogo_inegi_municipios.to_csv(
    OUTPUT_DIR / "catalogo_inegi_municipios_usados.csv",
    index=False,
    encoding="utf-8-sig"
)

✓ AGUASCALIENTES: 11 municipios descargados
✓ BAJA CALIFORNIA: 7 municipios descargados
✓ BAJA CALIFORNIA SUR: 5 municipios descargados
✓ CAMPECHE: 13 municipios descargados
✓ CDMX: 16 municipios descargados
✓ CHIAPAS: 124 municipios descargados
✓ CHIHUAHUA: 67 municipios descargados
✓ COAHUILA: 38 municipios descargados
✓ COLIMA: 10 municipios descargados
✓ DURANGO: 39 municipios descargados
✓ GUANAJUATO: 46 municipios descargados
✓ GUERRERO: 85 municipios descargados
✓ HIDALGO: 84 municipios descargados
✓ JALISCO: 125 municipios descargados
✓ MEXICO: 125 municipios descargados
✓ MICHOACAN: 113 municipios descargados
✓ MORELOS: 36 municipios descargados
✓ NAYARIT: 20 municipios descargados
✓ NUEVO LEON: 51 municipios descargados
✓ OAXACA: 570 municipios descargados
✓ PUEBLA: 217 municipios descargados
✓ QUERETARO: 18 municipios descargados
✓ QUINTANA ROO: 11 municipios descargados
✓ SAN LUIS POTOSI: 59 municipios descargados
✓ SINALOA: 20 municipios descargados
✓ SONORA: 72 municipios

,estado,cve_ent,cve_mun,cvegeo,municipio_inegi,cabecera_inegi,pob_total,estado_norm,municipio_inegi_norm
0,AGUASCALIENTES,01,001,01001,Aguascalientes,Aguascalientes,948990,AGUASCALIENTES,AGUASCALIENTES
1,AGUASCALIENTES,01,002,01002,Asientos,None,51536,AGUASCALIENTES,ASIENTOS
2,AGUASCALIENTES,01,003,01003,Calvillo,None,58250,AGUASCALIENTES,CALVILLO
3,AGUASCALIENTES,01,004,01004,Cosío,None,17000,AGUASCALIENTES,COSIO
4,AGUASCALIENTES,01,005,01005,Jesús María,None,129929,AGUASCALIENTES,JESUS MARIA


Shape: (2478, 9)


,estado,n_municipios
0,AGUASCALIENTES,11
1,BAJA CALIFORNIA,7
2,BAJA CALIFORNIA SUR,5
3,CAMPECHE,13
4,CDMX,16
5,CHIAPAS,124
6,CHIHUAHUA,67
7,COAHUILA,38
8,COLIMA,10
9,DURANGO,39


Municipios INEGI faltantes:
0


In [29]:
catalogo_inegi_municipios[
    (catalogo_inegi_municipios["estado_norm"] == "AGUASCALIENTES")
    & (catalogo_inegi_municipios["municipio_inegi_norm"] == "AGUASCALIENTES")
]

,estado,cve_ent,cve_mun,cvegeo,municipio_inegi,cabecera_inegi,pob_total,estado_norm,municipio_inegi_norm
0,AGUASCALIENTES,01,001,01001,Aguascalientes,Aguascalientes,948990,AGUASCALIENTES,AGUASCALIENTES


In [30]:
validacion_municipios = pares_estado_municipio.merge(
    catalogo_inegi_municipios[
        [
            "estado_norm",
            "municipio_inegi_norm",
            "municipio_inegi",
            "cve_ent",
            "cve_mun",
            "cvegeo"
        ]
    ],
    left_on=["estado_norm", "municipio_norm"],
    right_on=["estado_norm", "municipio_inegi_norm"],
    how="left"
)

validacion_municipios["existe_en_inegi"] = validacion_municipios["municipio_inegi"].notna()

resumen_validacion_municipios = (
    validacion_municipios
    .groupby(["escenario", "base", "existe_en_inegi"], dropna=False)
    .size()
    .reset_index(name="n")
)

no_encontrados = (
    validacion_municipios[
        ~validacion_municipios["existe_en_inegi"]
    ]
    .copy()
    .sort_values(["estado", "municipio"])
)

display(resumen_validacion_municipios)
display(no_encontrados)

validacion_municipios.to_csv(
    OUTPUT_DIR / "validacion_municipios_con_catalogo_inegi.csv",
    index=False,
    encoding="utf-8-sig"
)

no_encontrados.to_csv(
    OUTPUT_DIR / "municipios_no_encontrados_catalogo_inegi.csv",
    index=False,
    encoding="utf-8-sig"
)

,escenario,base,existe_en_inegi,n
0,ALTA,BANAVIM,False,6
1,ALTA,BANAVIM,True,465
2,ALTA,INE,False,3
3,ALTA,INE,True,37
4,ALTA_MEDIA,BANAVIM,False,8
5,ALTA_MEDIA,BANAVIM,True,1108
6,ALTA_MEDIA,INE,False,3
7,ALTA_MEDIA,INE,True,56


,estado,municipio,base,escenario,estado_norm,municipio_norm,cambio_detectable,municipio_inegi_norm,municipio_inegi,cve_ent,cve_mun,cvegeo,existe_en_inegi
15,CAMPECHE,CIUDAD DEL CARMEN,BANAVIM,ALTA,CAMPECHE,CIUDAD DEL CARMEN,False,NaN,NaN,NaN,NaN,NaN,False
533,CAMPECHE,CIUDAD DEL CARMEN,BANAVIM,ALTA_MEDIA,CAMPECHE,CIUDAD DEL CARMEN,False,NaN,NaN,NaN,NaN,NaN,False
16,CAMPECHE,SAN FRANCISCO DE CAMPECHE,BANAVIM,ALTA,CAMPECHE,SAN FRANCISCO DE CAMPECHE,False,NaN,NaN,NaN,NaN,NaN,False
535,CAMPECHE,SAN FRANCISCO DE CAMPECHE,BANAVIM,ALTA_MEDIA,CAMPECHE,SAN FRANCISCO DE CAMPECHE,False,NaN,NaN,NaN,NaN,NaN,False
536,CAMPECHE,XPUJIL,BANAVIM,ALTA_MEDIA,CAMPECHE,XPUJIL,False,NaN,NaN,NaN,NaN,NaN,False
40,CHIAPAS,CINTALAPA,BANAVIM,ALTA,CHIAPAS,CINTALAPA,False,NaN,NaN,NaN,NaN,NaN,False
576,CHIAPAS,CINTALAPA,BANAVIM,ALTA_MEDIA,CHIAPAS,CINTALAPA,False,NaN,NaN,NaN,NaN,NaN,False
89,COLIMA,CIUDAD DE ARMERIA,BANAVIM,ALTA,COLIMA,CIUDAD DE ARMERIA,False,NaN,NaN,NaN,NaN,NaN,False
698,COLIMA,CIUDAD DE ARMERIA,BANAVIM,ALTA_MEDIA,COLIMA,CIUDAD DE ARMERIA,False,NaN,NaN,NaN,NaN,NaN,False
90,COLIMA,CIUDAD DE VILLA DE ALVAREZ,BANAVIM,ALTA,COLIMA,CIUDAD DE VILLA DE ALVAREZ,False,NaN,NaN,NaN,NaN,NaN,False


In [33]:
# Preparar tabla de municipios no encontrados únicos con trazabilidad agregada

municipios_no_encontrados_unicos = (
    no_encontrados
    .groupby(["estado", "municipio", "estado_norm", "municipio_norm"], as_index=False)
    .agg({
        "base": lambda x: sorted(set(x)),
        "escenario": lambda x: sorted(set(x)),
    })
)

# Agregar columnas booleanas de presencia por base y escenario
municipios_no_encontrados_unicos["aparece_en_ine"] = (
    municipios_no_encontrados_unicos["base"].apply(lambda x: "INE" in x)
)

municipios_no_encontrados_unicos["aparece_en_banavim"] = (
    municipios_no_encontrados_unicos["base"].apply(lambda x: "BANAVIM" in x)
)

municipios_no_encontrados_unicos["aparece_en_alta"] = (
    municipios_no_encontrados_unicos["escenario"].apply(lambda x: "ALTA" in x)
)

municipios_no_encontrados_unicos["aparece_en_alta_media"] = (
    municipios_no_encontrados_unicos["escenario"].apply(lambda x: "ALTA_MEDIA" in x)
)

# Contar apariciones totales
apariciones = (
    no_encontrados
    .groupby(["estado", "municipio"])
    .size()
    .reset_index(name="n_apariciones")
)

municipios_no_encontrados_unicos = municipios_no_encontrados_unicos.merge(
    apariciones,
    on=["estado", "municipio"],
    how="left"
)

# Reordenar columnas
municipios_no_encontrados_unicos = municipios_no_encontrados_unicos[[
    "estado",
    "municipio",
    "estado_norm",
    "municipio_norm",
    "n_apariciones",
    "base",
    "escenario",
    "aparece_en_ine",
    "aparece_en_banavim",
    "aparece_en_alta",
    "aparece_en_alta_media"
]].sort_values(["estado", "municipio"])

display(municipios_no_encontrados_unicos)

# Guardar resultado
municipios_no_encontrados_unicos.to_csv(
    OUTPUT_DIR / "municipios_no_encontrados_unicos.csv",
    index=False,
    encoding="utf-8-sig"
)

print(f"\nTotal de municipios únicos no encontrados: {len(municipios_no_encontrados_unicos)}")
print(f"Guardado en: {OUTPUT_DIR / 'municipios_no_encontrados_unicos.csv'}")

,estado,municipio,estado_norm,municipio_norm,n_apariciones,base,escenario,aparece_en_ine,aparece_en_banavim,aparece_en_alta,aparece_en_alta_media
0,CAMPECHE,CIUDAD DEL CARMEN,CAMPECHE,CIUDAD DEL CARMEN,2,[BANAVIM],"[ALTA, ALTA_MEDIA]",False,True,True,True
1,CAMPECHE,SAN FRANCISCO DE CAMPECHE,CAMPECHE,SAN FRANCISCO DE CAMPECHE,2,[BANAVIM],"[ALTA, ALTA_MEDIA]",False,True,True,True
2,CAMPECHE,XPUJIL,CAMPECHE,XPUJIL,1,[BANAVIM],[ALTA_MEDIA],False,True,False,True
3,CHIAPAS,CINTALAPA,CHIAPAS,CINTALAPA,2,[BANAVIM],"[ALTA, ALTA_MEDIA]",False,True,True,True
4,COLIMA,CIUDAD DE ARMERIA,COLIMA,CIUDAD DE ARMERIA,2,[BANAVIM],"[ALTA, ALTA_MEDIA]",False,True,True,True
5,COLIMA,CIUDAD DE VILLA DE ALVAREZ,COLIMA,CIUDAD DE VILLA DE ALVAREZ,2,[BANAVIM],"[ALTA, ALTA_MEDIA]",False,True,True,True
6,NAYARIT,EL NAYAR,NAYARIT,EL NAYAR,2,[INE],"[ALTA, ALTA_MEDIA]",True,False,True,True
7,OAXACA,MAGDALENA APASCO,OAXACA,MAGDALENA APASCO,3,"[BANAVIM, INE]","[ALTA, ALTA_MEDIA]",True,True,True,True
8,OAXACA,SANTIAGO CHAZUMBA,OAXACA,SANTIAGO CHAZUMBA,2,[INE],"[ALTA, ALTA_MEDIA]",True,False,True,True
9,QUINTANA ROO,SOLIDARIDAD,QUINTANA ROO,SOLIDARIDAD,2,[BANAVIM],"[ALTA, ALTA_MEDIA]",False,True,True,True



Total de municipios únicos no encontrados: 10
Guardado en: ../Data/fusion_outputs/municipios_no_encontrados_unicos.csv


## Decisiones finales de validación geográfica

Después de validar los pares `estado + municipio` contra el catálogo oficial de INEGI, se revisaron manualmente los municipios que no coincidieron de forma exacta.

En un primer momento se consideró usar similitud textual para proponer correcciones automáticas. Sin embargo, se decidió no conservar esa etapa como parte definitiva del pipeline, porque la similitud entre nombres puede producir falsos positivos, especialmente en municipios con nombres parecidos, topónimos religiosos o nombres comunes.

En su lugar, se optó por una estrategia conservadora: aplicar únicamente correcciones manualmente justificadas y conservar sin modificar los casos donde el valor ya corresponde a un municipio oficial o donde no existe evidencia suficiente para corregirlo.

La revisión permitió distinguir distintos tipos de casos:

| Tipo de caso | Descripción | Decisión |
|---|---|---|
| `LOCALIDAD_A_MUNICIPIO` | El valor registrado corresponde a una ciudad, localidad o cabecera, pero el campo esperado es municipio. | Se corrige al municipio correspondiente. |
| `VARIANTE_A_OFICIAL` | El valor registrado es una variante o forma no oficial del nombre municipal. | Se corrige al nombre oficial. |
| `YA_ES_OFICIAL` | El valor registrado sí corresponde a un municipio oficial, aunque no haya coincidido inicialmente en la validación automática. | Se conserva sin cambios. |
| `NO_CORREGIR` | No hay evidencia suficiente para modificar el valor. | Se conserva o se excluye del matching geográfico fuerte, según el caso. |

Las correcciones validadas fueron:

| Estado | Valor original | Valor corregido | Tipo de decisión |
|---|---|---|---|
| `CAMPECHE` | `CIUDAD DEL CARMEN` | `CARMEN` | `LOCALIDAD_A_MUNICIPIO` |
| `CAMPECHE` | `SAN FRANCISCO DE CAMPECHE` | `CAMPECHE` | `LOCALIDAD_A_MUNICIPIO` |
| `CAMPECHE` | `XPUJIL` | `CALAKMUL` | `LOCALIDAD_A_MUNICIPIO` |
| `COLIMA` | `CIUDAD DE ARMERIA` | `ARMERIA` | `LOCALIDAD_A_MUNICIPIO` |
| `COLIMA` | `CIUDAD DE VILLA DE ALVAREZ` | `VILLA DE ALVAREZ` | `LOCALIDAD_A_MUNICIPIO` |
| `NAYARIT` | `EL NAYAR` | `DEL NAYAR` | `VARIANTE_A_OFICIAL` |

Los siguientes valores se conservaron sin cambios porque corresponden a municipios oficiales o no se justificó una corrección:

| Estado | Valor original | Decisión | Tipo de decisión |
|---|---|---|---|
| `CHIAPAS` | `CINTALAPA` | Conservar | `YA_ES_OFICIAL` |
| `OAXACA` | `MAGDALENA APASCO` | Conservar | `YA_ES_OFICIAL` |
| `OAXACA` | `SANTIAGO CHAZUMBA` | Conservar | `YA_ES_OFICIAL` |
| `QUINTANA ROO` | `SOLIDARIDAD` | Conservar | `YA_ES_OFICIAL` |

Esta decisión evita corregir municipios únicamente por parecido textual. Las correcciones se aplican solo cuando existe una justificación clara de que el valor capturado no corresponde al nivel geográfico esperado o usa una variante no oficial.

Después de aplicar estas correcciones, se debe repetir la validación contra INEGI y marcar qué registros tienen un par `estado + municipio` validado. Los registros que permanezcan sin validación no se eliminan automáticamente de las bases generales, pero no deben utilizarse como evidencia geográfica fuerte en el record linkage.

In [36]:
# Diccionario de correcciones validadas por (estado, municipio_original) -> municipio_corregido
correcciones_validadas = {
    ("CAMPECHE", "CIUDAD DEL CARMEN"): "CARMEN",
    ("CAMPECHE", "SAN FRANCISCO DE CAMPECHE"): "CAMPECHE",
    ("CAMPECHE", "XPUJIL"): "CALAKMUL",
    ("COLIMA", "CIUDAD DE ARMERIA"): "ARMERIA",
    ("COLIMA", "CIUDAD DE VILLA DE ALVAREZ"): "VILLA DE ALVAREZ",
    ("NAYARIT", "EL NAYAR"): "DEL NAYAR"
}

def aplicar_correcciones_municipios(df, nombre_base):
    """
    Aplica correcciones validadas de municipios a un dataframe.
    Conserva trazabilidad y crea columna de indicador de corrección.
    """
    df_corregido = df.copy()
    
    # Crear columna de municipio original si no existe
    if "municipio_original" not in df_corregido.columns:
        df_corregido["municipio_original"] = df_corregido["municipio"]
    
    # Crear columna de indicador de corrección
    df_corregido["municipio_corregido"] = False
    
    # Aplicar cada corrección validada
    for (estado_orig, municipio_orig), municipio_nuevo in correcciones_validadas.items():
        mask = (df_corregido["estado"] == estado_orig) & (df_corregido["municipio"] == municipio_orig)
        n_registros = mask.sum()
        
        if n_registros > 0:
            df_corregido.loc[mask, "municipio"] = municipio_nuevo
            df_corregido.loc[mask, "municipio_corregido"] = True
            print(f"  {nombre_base}: {estado_orig} - {municipio_orig} → {municipio_nuevo}: {n_registros} registros")
    
    # Recalcular estado_municipio
    df_corregido["estado_municipio"] = df_corregido["estado"] + " - " + df_corregido["municipio"]
    
    return df_corregido

# Aplicar correcciones a todas las bases
print("Aplicando correcciones validadas de municipios:\n")

print("INE Alta Confianza:")
ine_match_alta_export = aplicar_correcciones_municipios(ine_match_alta_export, "ine_match_alta_export")

print("\nBANAVIM Alta Confianza:")
bv_match_alta_export = aplicar_correcciones_municipios(bv_match_alta_export, "bv_match_alta_export")

print("\nINE Alta + Media Confianza:")
ine_match_alta_media_export = aplicar_correcciones_municipios(ine_match_alta_media_export, "ine_match_alta_media_export")

print("\nBANAVIM Alta + Media Confianza:")
bv_match_alta_media_export = aplicar_correcciones_municipios(bv_match_alta_media_export, "bv_match_alta_media_export")

# Resumen de correcciones por base
print("\n" + "="*80)
print("RESUMEN DE CORRECCIONES APLICADAS")
print("="*80)

resumen_correcciones = pd.DataFrame([
    {
        "base": "ine_match_alta_export",
        "n_registros_totales": len(ine_match_alta_export),
        "n_registros_corregidos": ine_match_alta_export["municipio_corregido"].sum()
    },
    {
        "base": "bv_match_alta_export",
        "n_registros_totales": len(bv_match_alta_export),
        "n_registros_corregidos": bv_match_alta_export["municipio_corregido"].sum()
    },
    {
        "base": "ine_match_alta_media_export",
        "n_registros_totales": len(ine_match_alta_media_export),
        "n_registros_corregidos": ine_match_alta_media_export["municipio_corregido"].sum()
    },
    {
        "base": "bv_match_alta_media_export",
        "n_registros_totales": len(bv_match_alta_media_export),
        "n_registros_corregidos": bv_match_alta_media_export["municipio_corregido"].sum()
    }
])

resumen_correcciones["porcentaje_corregidos"] = (
    resumen_correcciones["n_registros_corregidos"] / resumen_correcciones["n_registros_totales"] * 100
).round(2)

display(resumen_correcciones)

# Mostrar valores únicos corregidos por base
print("\n" + "="*80)
print("MUNICIPIOS CORREGIDOS (VALORES ÚNICOS) POR BASE")
print("="*80)

for df, nombre in [
    (ine_match_alta_export, "ine_match_alta_export"),
    (bv_match_alta_export, "bv_match_alta_export"),
    (ine_match_alta_media_export, "ine_match_alta_media_export"),
    (bv_match_alta_media_export, "bv_match_alta_media_export")
]:
    corregidos = df[df["municipio_corregido"]][
        ["estado", "municipio_original", "municipio"]
    ].drop_duplicates().sort_values(["estado", "municipio_original"])
    
    if len(corregidos) > 0:
        print(f"\n{nombre}:")
        display(corregidos)
    else:
        print(f"\n{nombre}: Sin correcciones")

Aplicando correcciones validadas de municipios:

INE Alta Confianza:

BANAVIM Alta Confianza:

INE Alta + Media Confianza:
  ine_match_alta_media_export: NAYARIT - EL NAYAR → DEL NAYAR: 1 registros

BANAVIM Alta + Media Confianza:
  bv_match_alta_media_export: CAMPECHE - CIUDAD DEL CARMEN → CARMEN: 31 registros
  bv_match_alta_media_export: CAMPECHE - SAN FRANCISCO DE CAMPECHE → CAMPECHE: 275 registros
  bv_match_alta_media_export: CAMPECHE - XPUJIL → CALAKMUL: 2 registros
  bv_match_alta_media_export: COLIMA - CIUDAD DE ARMERIA → ARMERIA: 9 registros
  bv_match_alta_media_export: COLIMA - CIUDAD DE VILLA DE ALVAREZ → VILLA DE ALVAREZ: 158 registros

RESUMEN DE CORRECCIONES APLICADAS


,base,n_registros_totales,n_registros_corregidos,porcentaje_corregidos
0,ine_match_alta_export,98,0,0.00
1,bv_match_alta_export,3824,0,0.00
2,ine_match_alta_media_export,128,1,0.78
3,bv_match_alta_media_export,34782,475,1.37



MUNICIPIOS CORREGIDOS (VALORES ÚNICOS) POR BASE

ine_match_alta_export: Sin correcciones

bv_match_alta_export: Sin correcciones

ine_match_alta_media_export:


,estado,municipio_original,municipio
2,NAYARIT,EL NAYAR,DEL NAYAR



bv_match_alta_media_export:


,estado,municipio_original,municipio
7848,CAMPECHE,CIUDAD DEL CARMEN,CARMEN
7088,CAMPECHE,SAN FRANCISCO DE CAMPECHE,CAMPECHE
464079,CAMPECHE,XPUJIL,CALAKMUL
15557,COLIMA,CIUDAD DE ARMERIA,ARMERIA
14655,COLIMA,CIUDAD DE VILLA DE ALVAREZ,VILLA DE ALVAREZ


In [37]:
# Reconstruir tabla de pares únicos estado + municipio después de correcciones
pares_estado_municipio_post_correccion = pd.concat([
    ine_match_alta_export[["estado", "municipio"]].drop_duplicates().assign(base="INE", escenario="ALTA"),
    bv_match_alta_export[["estado", "municipio"]].drop_duplicates().assign(base="BANAVIM", escenario="ALTA"),
    ine_match_alta_media_export[["estado", "municipio"]].drop_duplicates().assign(base="INE", escenario="ALTA_MEDIA"),
    bv_match_alta_media_export[["estado", "municipio"]].drop_duplicates().assign(base="BANAVIM", escenario="ALTA_MEDIA")
], ignore_index=True)

# Eliminar nulos
pares_estado_municipio_post_correccion = pares_estado_municipio_post_correccion.dropna(subset=["estado", "municipio"])

# Eliminar duplicados completos
pares_estado_municipio_post_correccion = pares_estado_municipio_post_correccion.drop_duplicates()

# Normalizar estado y municipio
pares_estado_municipio_post_correccion["estado_norm"] = pares_estado_municipio_post_correccion["estado"].apply(normalizar_geo)
pares_estado_municipio_post_correccion["municipio_norm"] = pares_estado_municipio_post_correccion["municipio"].apply(normalizar_geo)

# Ordenar
pares_estado_municipio_post_correccion = pares_estado_municipio_post_correccion.sort_values(
    ["escenario", "base", "estado", "municipio"]
).reset_index(drop=True)

# Validar contra catálogo INEGI
validacion_municipios_post_correccion = pares_estado_municipio_post_correccion.merge(
    catalogo_inegi_municipios[
        [
            "estado_norm",
            "municipio_inegi_norm",
            "municipio_inegi",
            "cve_ent",
            "cve_mun",
            "cvegeo"
        ]
    ],
    left_on=["estado_norm", "municipio_norm"],
    right_on=["estado_norm", "municipio_inegi_norm"],
    how="left"
)

validacion_municipios_post_correccion["existe_en_inegi"] = validacion_municipios_post_correccion["municipio_inegi"].notna()

# Resumen por escenario y base
resumen_validacion_municipios_post_correccion = (
    validacion_municipios_post_correccion
    .groupby(["escenario", "base", "existe_en_inegi"], dropna=False)
    .size()
    .reset_index(name="n")
)

# No encontrados después de correcciones
no_encontrados_post_correccion = (
    validacion_municipios_post_correccion[
        ~validacion_municipios_post_correccion["existe_en_inegi"]
    ]
    .copy()
    .sort_values(["estado", "municipio"])
)

# Comparación antes vs después
n_no_encontrados_antes = len(no_encontrados)
n_no_encontrados_despues = len(no_encontrados_post_correccion)
n_corregidos = n_no_encontrados_antes - n_no_encontrados_despues

print("="*80)
print("VALIDACIÓN GEOGRÁFICA POST-CORRECCIONES")
print("="*80)

print("\n--- RESUMEN POR ESCENARIO Y BASE ---")
display(resumen_validacion_municipios_post_correccion)

print("\n--- COMPARACIÓN ANTES Y DESPUÉS ---")
print(f"Municipios no encontrados ANTES: {n_no_encontrados_antes}")
print(f"Municipios no encontrados DESPUÉS: {n_no_encontrados_despues}")
print(f"Municipios CORREGIDOS: {n_corregidos}")
print(f"Reducción: {round(n_corregidos / n_no_encontrados_antes * 100, 2)}%")

print("\n--- MUNICIPIOS AÚN NO ENCONTRADOS ---")
display(no_encontrados_post_correccion)

VALIDACIÓN GEOGRÁFICA POST-CORRECCIONES

--- RESUMEN POR ESCENARIO Y BASE ---


,escenario,base,existe_en_inegi,n
0,ALTA,BANAVIM,False,2
1,ALTA,BANAVIM,True,469
2,ALTA,INE,False,2
3,ALTA,INE,True,38
4,ALTA_MEDIA,BANAVIM,False,3
5,ALTA_MEDIA,BANAVIM,True,1113
6,ALTA_MEDIA,INE,False,2
7,ALTA_MEDIA,INE,True,57



--- COMPARACIÓN ANTES Y DESPUÉS ---
Municipios no encontrados ANTES: 20
Municipios no encontrados DESPUÉS: 9
Municipios CORREGIDOS: 11
Reducción: 55.0%

--- MUNICIPIOS AÚN NO ENCONTRADOS ---


,estado,municipio,base,escenario,estado_norm,municipio_norm,municipio_inegi_norm,municipio_inegi,cve_ent,cve_mun,cvegeo,existe_en_inegi
40,CHIAPAS,CINTALAPA,BANAVIM,ALTA,CHIAPAS,CINTALAPA,NaN,NaN,NaN,NaN,NaN,False
576,CHIAPAS,CINTALAPA,BANAVIM,ALTA_MEDIA,CHIAPAS,CINTALAPA,NaN,NaN,NaN,NaN,NaN,False
484,OAXACA,MAGDALENA APASCO,INE,ALTA,OAXACA,MAGDALENA APASCO,NaN,NaN,NaN,NaN,NaN,False
1179,OAXACA,MAGDALENA APASCO,BANAVIM,ALTA_MEDIA,OAXACA,MAGDALENA APASCO,NaN,NaN,NaN,NaN,NaN,False
1648,OAXACA,MAGDALENA APASCO,INE,ALTA_MEDIA,OAXACA,MAGDALENA APASCO,NaN,NaN,NaN,NaN,NaN,False
495,OAXACA,SANTIAGO CHAZUMBA,INE,ALTA,OAXACA,SANTIAGO CHAZUMBA,NaN,NaN,NaN,NaN,NaN,False
1661,OAXACA,SANTIAGO CHAZUMBA,INE,ALTA_MEDIA,OAXACA,SANTIAGO CHAZUMBA,NaN,NaN,NaN,NaN,NaN,False
365,QUINTANA ROO,SOLIDARIDAD,BANAVIM,ALTA,QUINTANA ROO,SOLIDARIDAD,NaN,NaN,NaN,NaN,NaN,False
1350,QUINTANA ROO,SOLIDARIDAD,BANAVIM,ALTA_MEDIA,QUINTANA ROO,SOLIDARIDAD,NaN,NaN,NaN,NaN,NaN,False


In [38]:
# Agregar bandera municipio_validado_inegi a bases corregidas
# Fuente de verdad: validacion_municipios_post_correccion

# Crear tabla de lookup desde validacion_municipios_post_correccion
municipios_validados = validacion_municipios_post_correccion[
    validacion_municipios_post_correccion["existe_en_inegi"]
][["estado", "municipio"]].drop_duplicates().copy()

municipios_validados["municipio_validado_inegi"] = True

def agregar_bandera_municipio_validado(df, nombre_base):
    """
    Agrega bandera municipio_validado_inegi a un dataframe.
    Usa municipios_validados como tabla de lookup.
    """
    df_flagged = df.copy()
    
    # Merge left con municipios validados
    df_flagged = df_flagged.merge(
        municipios_validados,
        on=["estado", "municipio"],
        how="left"
    )
    
    # Rellenar False donde no hay validación
    df_flagged["municipio_validado_inegi"] = df_flagged["municipio_validado_inegi"].fillna(False)
    
    return df_flagged


# Aplicar bandera a todas las bases
print("Agregando bandera municipio_validado_inegi a bases corregidas:\n")

ine_match_alta_export = agregar_bandera_municipio_validado(ine_match_alta_export, "ine_match_alta_export")
bv_match_alta_export = agregar_bandera_municipio_validado(bv_match_alta_export, "bv_match_alta_export")
ine_match_alta_media_export = agregar_bandera_municipio_validado(ine_match_alta_media_export, "ine_match_alta_media_export")
bv_match_alta_media_export = agregar_bandera_municipio_validado(bv_match_alta_media_export, "bv_match_alta_media_export")

# Resumen por base y escenario
resumen_municipio_validado = pd.DataFrame([
    {
        "base": "ine_match_alta_export",
        "escenario": "ALTA",
        "registros_totales": len(ine_match_alta_export),
        "registros_validados": ine_match_alta_export["municipio_validado_inegi"].sum(),
        "registros_no_validados": (~ine_match_alta_export["municipio_validado_inegi"]).sum(),
        "porcentaje_validado": round(ine_match_alta_export["municipio_validado_inegi"].sum() / len(ine_match_alta_export) * 100, 2)
    },
    {
        "base": "bv_match_alta_export",
        "escenario": "ALTA",
        "registros_totales": len(bv_match_alta_export),
        "registros_validados": bv_match_alta_export["municipio_validado_inegi"].sum(),
        "registros_no_validados": (~bv_match_alta_export["municipio_validado_inegi"]).sum(),
        "porcentaje_validado": round(bv_match_alta_export["municipio_validado_inegi"].sum() / len(bv_match_alta_export) * 100, 2)
    },
    {
        "base": "ine_match_alta_media_export",
        "escenario": "ALTA_MEDIA",
        "registros_totales": len(ine_match_alta_media_export),
        "registros_validados": ine_match_alta_media_export["municipio_validado_inegi"].sum(),
        "registros_no_validados": (~ine_match_alta_media_export["municipio_validado_inegi"]).sum(),
        "porcentaje_validado": round(ine_match_alta_media_export["municipio_validado_inegi"].sum() / len(ine_match_alta_media_export) * 100, 2)
    },
    {
        "base": "bv_match_alta_media_export",
        "escenario": "ALTA_MEDIA",
        "registros_totales": len(bv_match_alta_media_export),
        "registros_validados": bv_match_alta_media_export["municipio_validado_inegi"].sum(),
        "registros_no_validados": (~bv_match_alta_media_export["municipio_validado_inegi"]).sum(),
        "porcentaje_validado": round(bv_match_alta_media_export["municipio_validado_inegi"].sum() / len(bv_match_alta_media_export) * 100, 2)
    }
])

display(resumen_municipio_validado)

print("\n" + "="*80)
print("VERIFICACIÓN: municipio_validado_inegi agregado a todas las bases")
print("="*80)

for df, nombre in [
    (ine_match_alta_export, "ine_match_alta_export"),
    (bv_match_alta_export, "bv_match_alta_export"),
    (ine_match_alta_media_export, "ine_match_alta_media_export"),
    (bv_match_alta_media_export, "bv_match_alta_media_export")
]:
    print(f"\n{nombre}:")
    print(f"  - ¿Tiene municipio_validado_inegi? {'municipio_validado_inegi' in df.columns}")
    print(f"  - Tipo: {df['municipio_validado_inegi'].dtype}")
    print(f"  - Valores únicos: {sorted(df['municipio_validado_inegi'].unique())}")

Agregando bandera municipio_validado_inegi a bases corregidas:



,base,escenario,registros_totales,registros_validados,registros_no_validados,porcentaje_validado
0,ine_match_alta_export,ALTA,98,95,3,96.94
1,bv_match_alta_export,ALTA,3824,3757,67,98.25
2,ine_match_alta_media_export,ALTA_MEDIA,128,125,3,97.66
3,bv_match_alta_media_export,ALTA_MEDIA,34782,34157,625,98.20



VERIFICACIÓN: municipio_validado_inegi agregado a todas las bases

ine_match_alta_export:
  - ¿Tiene municipio_validado_inegi? True
  - Tipo: bool
  - Valores únicos: [np.False_, np.True_]

bv_match_alta_export:
  - ¿Tiene municipio_validado_inegi? True
  - Tipo: bool
  - Valores únicos: [np.False_, np.True_]

ine_match_alta_media_export:
  - ¿Tiene municipio_validado_inegi? True
  - Tipo: bool
  - Valores únicos: [np.False_, np.True_]

bv_match_alta_media_export:
  - ¿Tiene municipio_validado_inegi? True
  - Tipo: bool
  - Valores únicos: [np.False_, np.True_]


## Record linkage INE--BANAVIM

Después de preparar las bases de matching, homologar la variable de vínculo y validar/corregir los campos geográficos, se inicia la etapa de record linkage entre INE y BANAVIM.

El objetivo de esta etapa no es afirmar automáticamente que un registro de INE y un registro de BANAVIM representan al mismo caso o a la misma persona. Dado que no existe un identificador común entre fuentes y BANAVIM no contiene el nombre del agresor, el resultado debe interpretarse como una generación de pares candidatos compatibles.

El primer escenario de vinculación será exacto y conservador. Se compararán registros que coincidan en:

- `estado_municipio`;
- `sexo`;
- `vinculo_match`.

Este primer escenario permite obtener pares candidatos de alta precisión, aunque probablemente con baja cobertura. La variable `municipio_validado_inegi` no se usará inicialmente para eliminar registros, sino como una marca de calidad geográfica que ayudará a interpretar la fuerza del candidato.

Posteriormente, si el escenario exacto resulta demasiado restrictivo, se podrá construir un escenario ampliado usando las bases de alta + media confianza o permitiendo una comparación menos estricta. Sin embargo, el primer paso será generar una línea base exacta y auditable.

In [39]:
# Crear pares candidatos exactos entre INE y BANAVIM (escenario alta confianza)

# Merge exacto por estado_municipio, sexo, vinculo_match
matches_exactos_alta = ine_match_alta_export.merge(
    bv_match_alta_export,
    on=["estado_municipio", "sexo", "vinculo_match"],
    how="inner",
    suffixes=("_ine", "_bv")
)

# Agregar columnas de contexto del matching
matches_exactos_alta["escenario_match"] = "ALTA_EXACTO"
matches_exactos_alta["tipo_match"] = "CANDIDATO_EXACTO"

# Reordenar columnas para mejor legibilidad
# Prioridad: IDs, criterios de matching, validación geográfica, trazabilidad

columnas_orden = [
    "escenario_match",
    "tipo_match",
    "id_ine_caso",
    "id_bv_caso",
    "estado_municipio",
    "sexo",
    "vinculo_match",
    "confianza_vinculo_match_ine",
    "confianza_vinculo_match_bv",
    "municipio_validado_inegi_ine",
    "municipio_validado_inegi_bv"
]

# Agregar columnas de trazabilidad disponibles
columnas_disponibles = [c for c in columnas_orden if c in matches_exactos_alta.columns]
columnas_faltantes = [c for c in matches_exactos_alta.columns if c not in columnas_orden]
columnas_faltantes = [c for c in columnas_faltantes if not c.startswith("vinculo_grupo")]

columnas_final = columnas_disponibles + sorted(columnas_faltantes)

matches_exactos_alta = matches_exactos_alta[columnas_final]

# Resumen
n_pares = len(matches_exactos_alta)

print("="*80)
print("PARES CANDIDATOS EXACTOS: ESCENARIO ALTA CONFIANZA")
print("="*80)
print(f"\nTotal de pares candidatos generados: {n_pares}")
print(f"\nCriterios de matching exacto:")
print("  - estado_municipio (coincidencia exacta)")
print("  - sexo (coincidencia exacta)")
print("  - vinculo_match (coincidencia exacta)")

print("\n" + "-"*80)
print("MUESTRA DE PARES CANDIDATOS")
print("-"*80)
display(matches_exactos_alta.head(10))

print("\n" + "-"*80)
print("ESTRUCTURA Y TIPOS DE DATOS")
print("-"*80)
print(matches_exactos_alta.info())

print("\n" + "-"*80)
print("ESTADÍSTICAS DESCRIPTIVAS")
print("-"*80)
resumen_matches_exactos_alta = pd.DataFrame([
    {
        "metrica": "Total de pares candidatos",
        "valor": n_pares
    },
    {
        "metrica": "Registros INE utilizados",
        "valor": len(ine_match_alta_export)
    },
    {
        "metrica": "Registros BANAVIM utilizados",
        "valor": len(bv_match_alta_export)
    },
    {
        "metrica": "Tasa de matching (pares / INE)",
        "valor": f"{round(n_pares / len(ine_match_alta_export) * 100, 2)}%"
    },
    {
        "metrica": "Pares con municipio validado (ambas fuentes)",
        "valor": (
            (matches_exactos_alta["municipio_validado_inegi_ine"]) & 
            (matches_exactos_alta["municipio_validado_inegi_bv"])
        ).sum()
    }
])

display(resumen_matches_exactos_alta)

PARES CANDIDATOS EXACTOS: ESCENARIO ALTA CONFIANZA

Total de pares candidatos generados: 8

Criterios de matching exacto:
  - estado_municipio (coincidencia exacta)
  - sexo (coincidencia exacta)
  - vinculo_match (coincidencia exacta)

--------------------------------------------------------------------------------
MUESTRA DE PARES CANDIDATOS
--------------------------------------------------------------------------------


,escenario_match,tipo_match,id_ine_caso,id_bv_caso,estado_municipio,sexo,vinculo_match,confianza_vinculo_match_ine,confianza_vinculo_match_bv,municipio_validado_inegi_ine,...,municipio_corregido_ine,municipio_fue_corregido_bv,municipio_fue_corregido_ine,municipio_ine,municipio_original_bv,municipio_original_ine,municipio_original_pre_correccion_bv,municipio_original_pre_correccion_ine,vinculo_victima_bv,vinculo_victima_ine
0,ALTA_EXACTO,CANDIDATO_EXACTO,INE_00042,BV_0051663,HIDALGO - ZIMAPAN,H,JERARQUICA_SUBORDINACION,ALTA,ALTA,True,...,False,False,False,ZIMAPAN,ZIMAPAN,ZIMAPAN,ZIMAPAN,ZIMAPAN,JEFE A O PATRON A,JERARQUICA
1,ALTA_EXACTO,CANDIDATO_EXACTO,INE_00055,BV_0790702,VERACRUZ - PEROTE,H,PARES,ALTA,ALTA,True,...,False,False,False,PEROTE,PEROTE,PEROTE,PEROTE,PEROTE,COMPANERO A,PARES
2,ALTA_EXACTO,CANDIDATO_EXACTO,INE_00074,BV_0571922,COAHUILA - FRANCISCO I MADERO,H,PARES,ALTA,ALTA,True,...,False,False,False,FRANCISCO I MADERO,FRANCISCO I MADERO,FRANCISCO I MADERO,FRANCISCO I MADERO,FRANCISCO I MADERO,COMPANERO A,PARES
3,ALTA_EXACTO,CANDIDATO_EXACTO,INE_00075,BV_0154241,MORELOS - TETELA DEL VOLCAN,H,JERARQUICA_SUBORDINACION,ALTA,ALTA,True,...,False,False,False,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,JEFE A O PATRON A,SUBORDINACION
4,ALTA_EXACTO,CANDIDATO_EXACTO,INE_00075,BV_0156652,MORELOS - TETELA DEL VOLCAN,H,JERARQUICA_SUBORDINACION,ALTA,ALTA,True,...,False,False,False,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,JEFE A O PATRON A,SUBORDINACION
5,ALTA_EXACTO,CANDIDATO_EXACTO,INE_00075,BV_0689665,MORELOS - TETELA DEL VOLCAN,H,JERARQUICA_SUBORDINACION,ALTA,ALTA,True,...,False,False,False,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,JEFE A O PATRON A,SUBORDINACION
6,ALTA_EXACTO,CANDIDATO_EXACTO,INE_00075,BV_0690933,MORELOS - TETELA DEL VOLCAN,H,JERARQUICA_SUBORDINACION,ALTA,ALTA,True,...,False,False,False,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,TETELA DEL VOLCAN,JEFE A O PATRON A,SUBORDINACION
7,ALTA_EXACTO,CANDIDATO_EXACTO,INE_00136,BV_0193398,PUEBLA - FRANCISCO Z MENA,H,JERARQUICA_SUBORDINACION,ALTA,ALTA,True,...,False,False,False,FRANCISCO Z MENA,FRANCISCO Z MENA,FRANCISCO Z MENA,FRANCISCO Z MENA,FRANCISCO Z MENA,JEFE A O PATRON A,JERARQUICA



--------------------------------------------------------------------------------
ESTRUCTURA Y TIPOS DE DATOS
--------------------------------------------------------------------------------
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 8 entries, 0 to 7
Data columns (total 33 columns):
 #   Column                                 Non-Null Count  Dtype 
---  ------                                 --------------  ----- 
 0   escenario_match                        8 non-null      object
 1   tipo_match                             8 non-null      object
 2   id_ine_caso                            8 non-null      object
 3   id_bv_caso                             8 non-null      object
 4   estado_municipio                       8 non-null      object
 5   sexo                                   8 non-null      object
 6   vinculo_match                          8 non-null      object
 7   confianza_vinculo_match_ine            8 non-null      object
 8   confianza_vinculo_match_bv       

,metrica,valor
0,Total de pares candidatos,8
1,Registros INE utilizados,98
2,Registros BANAVIM utilizados,3824
3,Tasa de matching (pares / INE),8.16%
4,Pares con municipio validado (ambas fuentes),8
